In [2]:
# ============================================================
#  06 Buildings / Morphology
#  数据源: 普查版本 (573K 栋, 带高度/层数/用途)
#  输出: 逐栋建筑 + 网格化形态指标
# ============================================================

from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import box
import warnings
warnings.filterwarnings("ignore")

RAW = Path("data_raw")
OUT = Path("data_out")
OUT.mkdir(exist_ok=True)

SHP = RAW / "Shenzen_Buildings_General/Shenzen_Buildings_General.shp"
BOUNDARY = Path("../04 Transport/data_raw/shenzhen_boundary.geojson")
BUILT_UP_2020 = Path("../00/16-城市建成区/16-城市建成区/深圳市_城市建成区2020.shp")

# ── 加载深圳边界 ──
shenzhen = gpd.read_file(BOUNDARY).to_crs(4326)
shenzhen_geom = shenzhen.union_all() if hasattr(shenzhen, "union_all") else shenzhen.unary_union

# ── 加载普查建筑 ──
print("Loading census buildings ...")
bldg_raw = gpd.read_file(SHP)
print(f"  Raw: {len(bldg_raw):,} buildings")

# ── 清洗 ──
bldg = bldg_raw.copy()

# 1) 只保留已竣工
bldg = bldg[bldg["BLDCOND"] == "已竣"]

# 2) 修复无效几何
bldg["geometry"] = bldg.geometry.make_valid()
bldg = bldg[~bldg.geometry.is_empty & bldg.geometry.notna()]

# 3) 过滤异常值: 面积 < 10 m² 或高度 < 0
bldg = bldg[(bldg["AREAF"] >= 10) & (bldg["BLDG_HEIGH"] >= 0)]

# 4) 裁剪到深圳边界
bldg = gpd.clip(bldg, shenzhen_geom)

# 5) 重命名列, 统一英文字段名
bldg = bldg.rename(columns={
    "BLDG_HEIGH": "height_m",
    "UP_BLDG_FL": "floors",
    "DOWN_BLDG_": "basement_floors",
    "AREAF": "footprint_m2",
    "BLDG_USAGE": "usage",
    "BLDSTRU": "structure",
    "NOWNAME": "name",
    "BLDADDR": "address",
})

# 6) 保留有用列
keep_cols = ["name", "address", "usage", "structure", "height_m", "floors",
             "basement_floors", "footprint_m2", "geometry"]
bldg = bldg[[c for c in keep_cols if c in bldg.columns]]

bldg = bldg.reset_index(drop=True)
print(f"  Cleaned: {len(bldg):,} buildings")
print(f"  Height: mean={bldg['height_m'].mean():.1f}m, max={bldg['height_m'].max():.0f}m")
print(f"  Area: mean={bldg['footprint_m2'].mean():.0f} m², median={bldg['footprint_m2'].median():.0f} m²")

Loading census buildings ...
  Raw: 573,007 buildings
  Cleaned: 567,309 buildings
  Height: mean=12.4m, max=442m
  Area: mean=1937 m², median=468 m²


In [3]:
# ============================================================
#  1. 逐栋派生指标
# ============================================================

# 高度分类 (对无人机飞行高度管制有意义)
bldg["height_class"] = pd.cut(
    bldg["height_m"],
    bins=[0, 3, 10, 24, 50, 100, 999],
    labels=["<3m", "3-10m", "10-24m", "24-50m", "50-100m", ">100m"],
    right=True,
)

bldg["is_tall"] = bldg["height_m"] > 100  # 超高层

# 用途大类映射 (原始分类太细, 归为大类便于可视化)
USAGE_MAP = {
    "私宅": "residential", "居住": "residential",
    "居住配套": "residential_support",
    "商服": "commercial", "办公": "commercial",
    "工业": "industrial", "仓储": "industrial",
    "公共设施": "public", "公共配套": "public",
    "交通": "transport", "市政": "public",
    "教育": "public", "医疗": "public",
}
bldg["usage_cat"] = bldg["usage"].map(USAGE_MAP).fillna("other")

# 体积近似 (footprint × height), 用于建筑密度计算
bldg["volume_m3"] = bldg["footprint_m2"] * bldg["height_m"]

print("=== Height Class Distribution ===")
print(bldg["height_class"].value_counts().sort_index().to_string())
print(f"\nSuper-tall (>100m): {bldg['is_tall'].sum()}")
print(f"\n=== Usage Category ===")
print(bldg["usage_cat"].value_counts().to_string())

=== Height Class Distribution ===
height_class
<3m        150576
3-10m      145125
10-24m     234906
24-50m      28842
50-100m      6884
>100m         914

Super-tall (>100m): 914

=== Usage Category ===
usage_cat
residential            402059
industrial             107073
commercial              21793
residential_support     18902
public                  12797
other                    3091
transport                1594


In [4]:
# ============================================================
#  1b. 内部深度 proxy (逐栋)
#  = 建筑轮廓内最大内切距离, 衡量"从边界到最深处有多远"
#  深度大 → 建筑内部配送路径长 → 无人机屋顶投递优势大
#  对 567K 栋全量算太慢, 采样计算后按 usage_cat 均值回填
# ============================================================
from shapely.ops import polylabel

SAMPLE_N = 5000

print(f"Calculating interior depth on {SAMPLE_N} sampled buildings ...")
sample = bldg.sample(n=min(SAMPLE_N, len(bldg)), random_state=42).copy()

depths = []
for idx, row in sample.iterrows():
    poly = row.geometry
    try:
        if not poly.is_valid:
            poly = poly.buffer(0)
        inner_pt = polylabel(poly, tolerance=0.5)
        depth = poly.boundary.distance(inner_pt)
        depths.append(depth)
    except Exception:
        depths.append(0)

sample["interior_depth_m"] = depths

# 按 usage_cat 计算均值, 回填到全量
depth_by_usage = sample.groupby("usage_cat")["interior_depth_m"].mean()
bldg["interior_depth_m"] = bldg["usage_cat"].map(depth_by_usage).fillna(depth_by_usage.mean())

# 对采样到的栋用精确值覆盖
bldg.loc[sample.index, "interior_depth_m"] = sample["interior_depth_m"]

print(f"\n=== Interior Depth by Usage ===")
print(depth_by_usage.round(2).to_string())
print(f"\nOverall mean: {bldg['interior_depth_m'].mean():.2f}m")

Calculating interior depth on 5000 sampled buildings ...

=== Interior Depth by Usage ===
usage_cat
commercial             0.0
industrial             0.0
other                  0.0
public                 0.0
residential            0.0
residential_support    0.0
transport              0.0

Overall mean: 0.00m


In [5]:
# ============================================================
#  2. 网格化形态指标
#  在深圳范围内建立正方形网格, 汇总建筑形态特征
#  供可视化页面做热力图 / 3D 柱状图
# ============================================================

GRID_SIZE_DEG = 0.005  # ~550m 网格, 可调: 0.01 (~1.1km), 0.002 (~220m)

minx, miny, maxx, maxy = shenzhen_geom.bounds
from shapely.prepared import prep as shapely_prep
sz_prep = shapely_prep(shenzhen_geom)

# 生成网格
grid_cells = []
grid_id = 0
x = minx
while x < maxx:
    y = miny
    while y < maxy:
        cell = box(x, y, x + GRID_SIZE_DEG, y + GRID_SIZE_DEG)
        if sz_prep.intersects(cell):
            grid_cells.append({"grid_id": grid_id, "geometry": cell})
            grid_id += 1
        y += GRID_SIZE_DEG
    x += GRID_SIZE_DEG

grid = gpd.GeoDataFrame(grid_cells, crs=4326)
print(f"Grid: {len(grid):,} cells ({GRID_SIZE_DEG}° ≈ {GRID_SIZE_DEG * 111:.0f}m)")

# 空间连接: 建筑 → 网格 (用建筑质心)
bldg_centroids = bldg.copy()
bldg_centroids["geometry"] = bldg_centroids.geometry.centroid
joined = gpd.sjoin(bldg_centroids, grid[["grid_id", "geometry"]], how="inner", predicate="within")

# 按网格汇总
grid_stats = joined.groupby("grid_id").agg(
    building_count=("height_m", "count"),
    avg_height=("height_m", "mean"),
    max_height=("height_m", "max"),
    height_std=("height_m", "std"),
    total_footprint=("footprint_m2", "sum"),
    total_volume=("volume_m3", "sum"),
    avg_floors=("floors", "mean"),
    n_tall=("is_tall", "sum"),
).reset_index()

# 补充: 建筑密度 = 总占地 / 网格面积
grid_area_m2 = (GRID_SIZE_DEG * 111320) ** 2
grid_stats["building_density"] = grid_stats["total_footprint"] / grid_area_m2
grid_stats["volume_density"] = grid_stats["total_volume"] / grid_area_m2

# 主导用途 (众数)
dominant = joined.groupby("grid_id")["usage_cat"].agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else "other")
dominant.name = "dominant_usage"
grid_stats = grid_stats.merge(dominant, on="grid_id", how="left")

# 填充无建筑的网格
grid_stats["height_std"] = grid_stats["height_std"].fillna(0)

# 合并回网格几何
grid_result = grid.merge(grid_stats, on="grid_id", how="left")
grid_result = grid_result.fillna(0)
grid_result.loc[grid_result["dominant_usage"] == 0, "dominant_usage"] = "empty"

# 用途多样性 (Shannon 熵): 多种用途混合 → 全天活动强 → overload 风险
from scipy.stats import entropy as shannon_entropy
usage_pivot = joined.groupby(["grid_id", "usage_cat"]).size().unstack(fill_value=0)
diversity = usage_pivot.apply(lambda r: shannon_entropy(r[r > 0]), axis=1)
diversity.name = "usage_diversity"
grid_stats = grid_stats.merge(diversity, on="grid_id", how="left")
grid_stats["usage_diversity"] = grid_stats["usage_diversity"].fillna(0)

# 合并回网格几何
grid_result = grid.merge(grid_stats, on="grid_id", how="left")
grid_result = grid_result.fillna(0)
grid_result.loc[grid_result["dominant_usage"] == 0, "dominant_usage"] = "empty"

print(f"Grids with buildings: {(grid_result['building_count'] > 0).sum():,} / {len(grid_result):,}")
print(f"Avg building density: {grid_result.loc[grid_result['building_count'] > 0, 'building_density'].mean():.3f}")
print(f"Max height in any grid: {grid_result['max_height'].max():.0f}m")
print(f"Usage diversity: mean={grid_result.loc[grid_result['building_count'] > 0, 'usage_diversity'].mean():.3f}")

Grid: 7,392 cells (0.005° ≈ 1m)
Grids with buildings: 5,180 / 7,392
Avg building density: 0.685
Max height in any grid: 442m
Usage diversity: mean=0.708


In [6]:
# ============================================================
#  3. 建成区校验 + 保存输出 + 统计摘要
# ============================================================

# ── 建成区 2020 校验 ──
if BUILT_UP_2020.exists():
    built_up = gpd.read_file(BUILT_UP_2020).to_crs(4326)
    built_up_geom = built_up.union_all() if hasattr(built_up, "union_all") else built_up.unary_union
    from shapely.prepared import prep as _prep
    bu_prep = _prep(built_up_geom)
    in_built_up = bldg.geometry.centroid.apply(lambda pt: bu_prep.contains(pt))
    bldg["in_built_up_area"] = in_built_up
    print(f"Built-up area check: {in_built_up.sum():,} / {len(bldg):,} buildings inside ({in_built_up.mean()*100:.1f}%)")
    # 也标注到网格
    grid_result["in_built_up_area"] = grid_result.geometry.centroid.apply(lambda pt: bu_prep.contains(pt))
    del built_up, built_up_geom, bu_prep
else:
    bldg["in_built_up_area"] = True
    grid_result["in_built_up_area"] = True
    print("Built-up area shp not found, skipping")

# 逐栋建筑
bldg.to_file(OUT / "sz_buildings.gpkg", driver="GPKG")
print(f"Buildings -> data_out/sz_buildings.gpkg  ({len(bldg):,} rows)")

# 网格化形态
grid_result.to_file(OUT / "sz_building_grid.gpkg", driver="GPKG")
print(f"Grid      -> data_out/sz_building_grid.gpkg  ({len(grid_result):,} cells)")

# ── 统计摘要 ──
print("\n=== Building Summary ===")
print(f"Total buildings: {len(bldg):,}")
print(f"Height: mean={bldg['height_m'].mean():.1f}m  median={bldg['height_m'].median():.1f}m  max={bldg['height_m'].max():.0f}m")
print(f"Floors: mean={bldg['floors'].mean():.1f}  max={bldg['floors'].max():.0f}")
print(f"Super-tall (>100m): {bldg['is_tall'].sum()}")

print("\n=== Height Distribution ===")
print(bldg["height_class"].value_counts().sort_index().to_string())

print("\n=== Usage Distribution ===")
print(bldg["usage_cat"].value_counts().to_string())

print("\n=== Grid Morphology (non-empty grids) ===")
g = grid_result[grid_result["building_count"] > 0]
print(f"Building density: mean={g['building_density'].mean():.3f}  max={g['building_density'].max():.3f}")
print(f"Volume density:   mean={g['volume_density'].mean():.1f}  max={g['volume_density'].max():.1f}")
print(f"Dominant usage distribution:")
print(g["dominant_usage"].value_counts().to_string())

Built-up area check: 546,949 / 567,309 buildings inside (96.4%)
Buildings -> data_out/sz_buildings.gpkg  (567,309 rows)
Grid      -> data_out/sz_building_grid.gpkg  (7,392 cells)

=== Building Summary ===
Total buildings: 567,309
Height: mean=12.4m  median=9.0m  max=442m
Floors: mean=4.1  max=115
Super-tall (>100m): 914

=== Height Distribution ===
height_class
<3m        150576
3-10m      145125
10-24m     234906
24-50m      28842
50-100m      6884
>100m         914

=== Usage Distribution ===
usage_cat
residential            402059
industrial             107073
commercial              21793
residential_support     18902
public                  12797
other                    3091
transport                1594

=== Grid Morphology (non-empty grids) ===
Building density: mean=0.685  max=6.651
Volume density:   mean=28.2  max=1175.9
Dominant usage distribution:
dominant_usage
residential            2926
industrial             1515
public                  342
commercial              225
o